# Study Notes: Instruction Fine-Tuning in Large Language Models

## 1. Introduction: The Limitation of Base Models

Pre-trained language models (often referred to as **base models**) are trained on vast amounts of internet text using a **next-word prediction** (causal language modeling) objective. While they learn substantial world knowledge and language syntax, they have a critical limitation: **they do not inherently understand or follow user instructions.**

* **Example:** If prompted with *"What is the national flower of India?"*, a base model might generate *"What is the national animal of India?"*
* **Why does this happen?** The model is merely predicting the most probable next sequence of words based on its training data (e.g., it might think it is generating a list of trivia questions found on a web page). Next-word prediction does not guarantee instruction adherence.

## 2. Classical Multitask Learning vs. Instruction Tuning

To understand instruction tuning, we first look at its predecessor: Multitask Learning.

### Classical Multitask Learning

* **Mechanism:** The model is trained on multiple tasks (e.g., sentiment analysis, machine translation) simultaneously by simply passing input-output pairs.
* **The Flaw:** The model is never explicitly told *which* task it is solving. When faced with an entirely new, unseen task during inference (e.g., COLA - Corpus of Linguistic Acceptability), the model is clueless and fails to generalize because it lacks context for what it is supposed to do.

### Instruction Tuning (Instruction Fine-Tuning)

* **Mechanism:** We describe the tasks using **natural language instructions**. During training, the model receives the Instruction + Input + Target Output.
* **Example:** `[Instruction]: Translate the sentence to Hindi.` + `[Input]: It has been raining.` + `[Target]: ...`
* **The Advantage (Zero-Shot Generalization):** Because the model learns to map natural language instructions to specific actions, it can generalize to **unseen tasks** during inference, provided the new task is also described via a natural language instruction.

> **Key Exam Distinction:**
> * **Instruction Tuning:** Involves updating the model's actual weights/parameters via fine-tuning.
> * **Prompting / In-Context Learning:** Does **not** update any model parameters; it only leverages the pre-trained weights at inference time.
> 
> 

---

## 3. Training Mechanism & Architecture

Instruction tuning is implemented as a supervised fine-tuning step. The model is trained using **cross-entropy loss** on the generated tokens.

### Teacher Forcing

During training, instead of feeding the model's own predictions back into it to generate the next word, we feed the **actual target output (ground truth)**. This technique is called **teacher forcing** and stabilizes/speeds up training.

### Loss Formulation

For a sequence, the model predicts the probability of the next token $o_t$ conditioned on the instruction/input sequence $i$ and all previously generated target tokens $o_{<t}$. We train by maximizing the Log-Likelihood:

$$\mathcal{L} = \sum_{t=1}^{N} \log P(o_t \vert{} i_1, i_2, \dots, i_k, o_1, o_2, \dots, o_{t-1})$$

### Architectural Differences

1. **Decoder-only Models (e.g., GPT, PaLM):** The instruction, input, and output are all concatenated and fed into the decoder. Causal masking ensures tokens only attend to previous tokens.
2. **Encoder-Decoder Models (e.g., T5):** The instruction and input are fed into the **encoder**, while the output is generated by the **decoder** (which also uses teacher forcing during training).

---

## 4. Scaling Instruction Fine-Tuned Models

Research (e.g., the FLAN papers) demonstrates how to scale instruction tuning for better performance.

* **Task Generation:** A single dataset can yield multiple tasks. For example, the SQuAD dataset (originally for extractive Question Answering) can be repurposed into:
* Standard QA.
* Query generation (generate a question given the answer and context).
* Context generation.


* **Instruction Diversity (Templates):** For a single task (e.g., Natural Language Inference), researchers create multiple phrasing templates (e.g., 4 to 10 variations of asking the same thing). Training on varied phrasings heavily improves the model's robustness and generalization.
* **Data Formats Evaluated:** Models are trained on various prompt structures to handle different user inputs:
1. Zero-shot (no examples).
2. Few-shot (with examples).
3. Chain of Thought (CoT - asking the model to "reason step-by-step").
4. CoT with examples.


* **Scaling Laws & Compute Cost:**
* Performance heavily increases when **model size (parameters)** and the **number of fine-tuning tasks** increase.
* **Cost Efficiency:** The computational cost of instruction fine-tuning is extremely low—typically only **0.2% to 1.6%** of the compute required for the original pre-training phase.


* **Generalizability:** Instruction tuning works across different architectures (Encoder-Decoder vs. Decoder-only) and different pre-training objectives (e.g., Span Corruption for T5, Causal LM for PaLM).

---

## 5. Automated Data Generation: "Self-Instruct"

Creating human-written instruction datasets is tedious and unscalable. The **Self-Instruct** framework uses an existing powerful LLM (like GPT-3) to automatically generate instruction tuning data.

### The Self-Instruct Pipeline

1. **Seed Tasks:** Start with a small human-written pool of tasks (e.g., 175 seed tasks).
2. **Instruction Generation:** Sample 8 tasks (6 human + 2 LLM-generated) and prompt the LLM to generate *new* instructions based on these examples.
3. **Task Classification:** Prompt the LLM (via few-shot learning) to classify whether the newly generated instruction is a **classification task** (finite output space, e.g., Yes/No, Positive/Negative) or a **non-classification task** (open-ended, e.g., story writing).
4. **Instance Generation (Crucial Distinction):**
* **Non-Classification Tasks:** Use the standard **Input-first approach** (Instruction $\rightarrow$ LLM generates Input $\rightarrow$ LLM generates Output).
* **Classification Tasks:** Use an **Output-first approach**. *Why?* Because LLMs tend to heavily favor majority class labels, leading to imbalanced datasets. By giving the LLM the target class label first and asking it to generate a corresponding input, we guarantee balanced class representation.


5. **Filtering Mechanisms:**
* **ROUGE-L Similarity:** Compare new instructions to existing ones. If the ROUGE-L score is **$\geq 0.7$**, the new instruction is discarded to maintain diversity.
* **Keyword Exclusion:** Remove instructions asking for non-text modalities (e.g., containing words like "image", "picture", "graph").
* **Deduplication:** Remove exact duplicate instances, or instances that share the exact same input but have conflicting outputs.
* **Heuristics:** Filter out instructions that are too long or too short.



---

## 6. Future Extensions (Brief Mention)

While instruction tuning teaches a model to follow commands, other techniques further refine behavior:

* **RLHF (Reinforcement Learning from Human Feedback):** Aligns the instruction-tuned model with human preferences and safety bounds.
* **PEFT (Parameter-Efficient Fine-Tuning):** Because fully updating all model weights is memory-intensive, techniques like LoRA are used to achieve similar results by updating only a small fraction of parameters.

---

## Practice Questions (NPTEL Style)

**Q1. What is the primary difference between Instruction Fine-Tuning and In-Context Learning (Prompting)?**
A) Prompting requires more compute than Instruction Fine-Tuning.
B) Instruction Fine-Tuning updates the model's parameters, whereas Prompting does not.
C) Prompting uses teacher forcing, while Instruction Fine-Tuning uses standard autoregressive generation.
D) Instruction Fine-Tuning can only be used on Encoder-Decoder architectures.
*Answer: B. Instruction tuning is a supervised learning phase that physically updates the model's weights. Prompting simply feeds text at inference time without updating weights.*

**Q2. During the Self-Instruct data generation pipeline, why is the "Output-first" approach used specifically for classification tasks?**
A) Because classification tasks require longer contexts.
B) To reduce the ROUGE-L similarity score.
C) To prevent the LLM from heavily favoring popular/majority class labels, ensuring a balanced dataset.
D) Because the loss function cannot process classification inputs otherwise.
*Answer: C. The output-first approach forces the LLM to generate inputs for all possible class labels evenly, preventing class imbalance in the training data.*

**Q3. Which of the following statements about the computational cost of Instruction Fine-Tuning is correct?**
A) It requires roughly the same amount of compute as pre-training.
B) It requires significantly more compute than pre-training due to the complex templates.
C) It requires a very small fraction of compute (typically < 2%) compared to pre-training.
D) It cannot be calculated because it depends entirely on the ROUGE-L score.
*Answer: C. The lecture explicitly notes the compute for fine-tuning is very low (0.2% - 1.6%) compared to pre-training.*

**Q4. In the filtering stage of the Self-Instruct framework, what threshold of ROUGE-L similarity is typically used to discard a newly generated instruction?**
A) $\geq$ 0.3
B) $\geq$ 0.5
C) $\geq$ 0.7
D) $\geq$ 0.9
*Answer: C. Instructions with a ROUGE-L similarity score of 0.7 or higher compared to existing instructions are discarded to maintain dataset diversity.*

**Q5. When training an Encoder-Decoder model (like T5) via instruction tuning, where are the instruction and input sequences fed?**
A) Both are fed exclusively to the decoder.
B) Only the instruction is fed to the encoder, and the input is fed to the decoder.
C) Both the instruction and the input are fed to the encoder.
D) They are concatenated with the output and fed to a classification head.
*Answer: C. In encoder-decoder models, the instruction and input are processed by the encoder, while the target output is generated by the decoder.*

---

## Revision Summary

* **Base Models:** Do next-word prediction; they do not inherently follow instructions.
* **Instruction Tuning:** Fine-tunes models on `(Instruction, Input, Output)` tuples, updating model weights via cross-entropy loss (maximizing log-likelihood) using teacher forcing.
* **Generalization:** By describing tasks in natural language, models can perform **zero-shot** on entirely new tasks they didn't see in training.
* **Scaling:** More model parameters + more instruction tasks = drastically better performance. Compute cost is surprisingly low ($<2\%$ of pre-training).
* **Instruction Diversity:** Using multiple prompt templates for the same task prevents overfitting and improves robustness.
* **Self-Instruct:** Automates dataset creation using an LLM.
* Uses *Few-shot prompting* to classify task types.
* Uses *Output-first generation* for classification tasks to balance labels.
* *Filters* rigorously (ROUGE-L $<0.7$, drops non-text keywords, drops dupes).